# 音のプログラミング 第2回: エンベロープとADSR

**学習目標:**
- エンベロープ（音量変化）の概念を理解する
- ADSRエンベロープの4つの段階を学ぶ
- 楽器らしい自然な音を作る
- クリック音の問題と解決方法を体験する

**所要時間:** 90分

## 🛠️ 環境設定

In [ ]:
# 🛠️ 環境セットアップ

# 共通ライブラリのインポート
import sys
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

# Google Colab環境かどうかを判定
try:
    import google.colab
    IN_COLAB = True
    print("🔧 Google Colab環境で実行中...")
except ImportError:
    IN_COLAB = False
    print("🏠 ローカル環境で実行中")

# ライブラリのセットアップ
if IN_COLAB:
    print("🔧 Google Colab環境を設定中...")
    
    # 必要なパッケージをインストール
    !pip install japanize-matplotlib
    
    # GitHubからライブラリをクローン
    !git clone https://github.com/ggszk/simple-audio-programming.git
    
    # パスを追加
    sys.path.append('/content/simple-audio-programming')

    # ライブラリ設定
    import japanize_matplotlib
    
else:
    print("🔧 ローカル環境を設定中...")

    # パスを追加
    sys.path.append('..')

    # 日本語フォント設定（Mac）
    import platform
    if platform.system() == 'Darwin':
        plt.rcParams['font.family'] = 'Hiragino Sans'
    else:
        plt.rcParams['font.family'] = 'Meiryo'

print("\n🎵 Simple Audio Programming へようこそ！")
print("✅ セットアップ完了")

## 🛠️ 今回使用するライブラリ

In [ ]:
from audio_lib import sine_wave, adsr, linear_envelope, save_audio, AudioSignal
from audio_lib.synthesis import note_to_frequency, note_name_to_number
from audio_lib.notebook import play_sound, plot_waveform

print("🛠️ ライブラリを読み込みました")

## 🚨 問題: クリック音を体験しよう

前回作ったサイン波には実は問題があります。聞いてみましょう！

💡 **ポイント**: サイン波は `sin(0) = 0` から始まるため、冒頭では無音からなめらかに立ち上がりクリック音は起きにくいです。一方、終了時は波形がちょうど0の位置で終わるとは限らないため、0以外の値から突然無音になる不連続が生じ、クリック音として聞こえます。

In [ ]:
# 前回と同じようにサイン波を作る
frequency = 440
duration = 1.0

raw_signal = sine_wave(frequency, duration)

print(f"📊 信号の急激な変化: 開始値={raw_signal.data[0]:.3f}, 終了値={raw_signal.data[-1]:.3f}")

# ファイル保存でクリック音を保持
save_audio("signal_without_envelope.wav", raw_signal)
print("📁 クリック音確認用: signal_without_envelope.wav を保存しました")

audio_player = play_sound(raw_signal, "エンベロープなし（クリック音あり）")
display(audio_player)

## 🎵 エンベロープとは？

### 現実の楽器の特徴
- **ピアノ**: 鍵盤を押した瞬間から音が鳴り、徐々に小さくなる
- **バイオリン**: 弓を当てて徐々に音が大きくなり、弓を離すと消える
- **フルート**: 息を吹き始めてから安定し、息を止めると消える

→ **音量が時間とともに変化する！**

### エンベロープ（包絡線）
音量の時間的変化を制御する仕組み

## 🎯 実習1: シンプルなフェードイン・フェードアウト

In [ ]:
# シンプルなフェードイン・フェードアウトエンベロープ
envelope_data = linear_envelope(
    duration,
    fade_in=0.1,   # 0.1秒かけてフェードイン
    fade_out=0.1,  # 0.1秒かけてフェードアウト
)

# エンベロープの形を見てみよう
time_array = np.linspace(0, duration, len(envelope_data))

plt.figure(figsize=(12, 6))
plt.plot(time_array, envelope_data.data, 'r-', linewidth=3, label='エンベロープ')
plt.title('リニアエンベロープ（フェードイン・フェードアウト）', fontsize=16)
plt.xlabel('時間 (秒)', fontsize=12)
plt.ylabel('音量', fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

これがエンベロープの形です。音の始まりと終わりがなめらかになっています。

In [ ]:
# エンベロープを音に適用
signal_with_envelope = raw_signal * envelope_data

display(play_sound(signal_with_envelope, "エンベロープ適用後"))

クリック音が消えて、自然な音になりました！

### エンベロープ適用前後の波形を比較してみよう

In [ ]:
plot_waveform(raw_signal, duration=0.1, title="オリジナル")
plot_waveform(signal_with_envelope, duration=0.1, title="適用後")

## 🎹 ADSR エンベロープ

楽器の音をもっとリアルに表現するための4段階エンベロープ

### 4つの段階
1. **Attack (アタック)**: 音の立ち上がり
2. **Decay (ディケイ)**: 最大音量から減衰
3. **Sustain (サステイン)**: 一定音量の維持
4. **Release (リリース)**: 音の消失

```
音量
 ↑
 |     /\     
 |    /  \    
 |   /    \___________
 |  /                 \
 | /                   \
 |/                     \
 +--A--D----S------R----→ 時間
```

## 🎯 実習2: ADSRエンベロープを体験しよう

In [ ]:
# 基本的なADSRエンベロープ
duration = 2.0  # 2秒の音
adsr_data = adsr(
    duration,
    attack=0.1,    # 0.1秒でアタック
    decay=0.2,     # 0.2秒でディケイ
    sustain=0.7,   # 70%のレベルでサステイン
    release=0.5,   # 0.5秒でリリース
)

# ADSRエンベロープの形を可視化
time_array = np.linspace(0, duration, len(adsr_data))

plt.figure(figsize=(14, 8))
plt.plot(time_array, adsr_data.data, 'g-', linewidth=3, label='ADSR エンベロープ')

# 各段階に注釈を追加
plt.axvline(x=0.1, color='r', linestyle='--', alpha=0.7, label='Attack終了')
plt.axvline(x=0.3, color='orange', linestyle='--', alpha=0.7, label='Decay終了')
plt.axvline(x=1.5, color='purple', linestyle='--', alpha=0.7, label='Release開始')

plt.title('ADSR エンベロープ', fontsize=16)
plt.xlabel('時間 (秒)', fontsize=12)
plt.ylabel('音量', fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
# ADSRエンベロープを音に適用
frequency = 440
signal = sine_wave(frequency, duration)
adsr_signal = signal * adsr_data

display(play_sound(adsr_signal, "ADSR エンベロープ付き"))

まるでピアノやシンセサイザーのような音になりました！

## 🎯 実習3: ADSRパラメータを変えて楽器を模倣しよう

In [ ]:
# ピアノらしい音（速いアタック、長いリリース）
duration = 3.0
piano_envelope = adsr(
    duration,
    attack=0.01,   # 瞬間的なアタック
    decay=0.3,     # やや長いディケイ
    sustain=0.3,   # 低めのサステイン
    release=1.5,   # 長いリリース
)

signal = sine_wave(440, duration)
piano_sound = signal * piano_envelope

display(play_sound(piano_sound, "ピアノ"))

In [ ]:
# オルガンらしい音（速いアタック、サステイン重視）
organ_envelope = adsr(
    duration,
    attack=0.05,   # 短いアタック
    decay=0.0,     # ディケイなし
    sustain=1.0,   # 最大レベルでサステイン
    release=0.1,   # 短いリリース
)

organ_sound = signal * organ_envelope

display(play_sound(organ_sound, "オルガン"))

In [ ]:
# 弦楽器らしい音（ゆっくりなアタック）
string_envelope = adsr(
    duration,
    attack=0.3,    # ゆっくりなアタック
    decay=0.1,     # 短いディケイ
    sustain=0.8,   # 高めのサステイン
    release=0.4,   # 中程度のリリース
)

string_sound = signal * string_envelope

display(play_sound(string_sound, "弦楽器"))

💡 同じサイン波でも、エンベロープで全く違う楽器に聞こえます！

## 🎯 実習4: エンベロープ付きメロディーを作ろう

In [ ]:
# きらきら星のメロディー（最初の部分）
melody_notes = [
    ('C4', 0.5), ('C4', 0.5), ('G4', 0.5), ('G4', 0.5),
    ('A4', 0.5), ('A4', 0.5), ('G4', 1.0),
    ('F4', 0.5), ('F4', 0.5), ('E4', 0.5), ('E4', 0.5),
    ('D4', 0.5), ('D4', 0.5), ('C4', 1.0)
]

# メロディーを生成
melody_audio_data = []

for note_name, note_duration in melody_notes:
    midi_number = note_name_to_number(note_name)
    frequency = note_to_frequency(midi_number)
    note_signal = sine_wave(frequency, note_duration)
    note_envelope = adsr(note_duration, attack=0.01, decay=0.2, sustain=0.4, release=0.3)
    note_with_envelope = note_signal * note_envelope
    melody_audio_data.append(note_with_envelope.data)

# 全ての音をつなげる
full_melody = np.concatenate(melody_audio_data)
melody_signal = AudioSignal(full_melody, 44100)

melody_audio = play_sound(melody_signal, "きらきら星（エンベロープ付き）")
display(melody_audio)

エンベロープのおかげで、各音符が自然につながった美しいメロディーができました。

## 🎯 実習5: エンベロープありなしの比較

In [ ]:
# 同じメロディーをエンベロープなしで作る
melody_without_envelope = []

for note_name, note_duration in melody_notes:
    midi_number = note_name_to_number(note_name)
    frequency = note_to_frequency(midi_number)
    note_signal = sine_wave(frequency, note_duration)
    melody_without_envelope.append(note_signal.data)

melody_raw = np.concatenate(melody_without_envelope)
raw_signal_melody = AudioSignal(melody_raw, 44100)

display(play_sound(raw_signal_melody, "エンベロープなし（クリック音あり）"))
display(melody_audio)

💡 **比較してみてください：**
- エンベロープなし：各音符の境界でクリック音
- エンベロープあり：なめらかで自然な音楽

## 🏆 チャレンジ課題

In [ ]:
# チャレンジ1: あなただけのエンベロープを作ろう
# パラメータを変更して、理想の音を見つけてください

# テスト音を作成
test_duration = 2.0
test_signal = sine_wave(440, test_duration)
my_envelope = adsr(
    test_duration,
    attack=0.1,     # ここを変更してください (0.01～1.0)
    decay=0.2,      # ここを変更してください (0.0～1.0)
    sustain=0.5,    # ここを変更してください (0.0～1.0)
    release=0.3,    # ここを変更してください (0.1～2.0)
)
my_sound = test_signal * my_envelope

# エンベロープの形を表示
time_array = np.linspace(0, test_duration, len(my_envelope))
plt.figure(figsize=(12, 6))
plt.plot(time_array, my_envelope.data, 'b-', linewidth=3)
plt.title('あなたのオリジナルエンベロープ', fontsize=16)
plt.xlabel('時間 (秒)')
plt.ylabel('音量')
plt.grid(True, alpha=0.3)
plt.show()

display(play_sound(my_sound, "オリジナル音色"))

In [ ]:
# チャレンジ2: 短いメロディーを作ろう
# 好きな音符を並べてください

# 使える音名: C4, D4, E4, F4, G4, A4, B4, C5
my_melody = [
    ('C4', 0.5),   # ド
    ('E4', 0.5),   # ミ
    ('G4', 0.5),   # ソ
    ('C5', 1.0),   # 高いド
    # ここに音符を追加してください
]

# あなたのエンベロープを使用
my_melody_audio = []

for note_name, note_duration in my_melody:
    midi_number = note_name_to_number(note_name)
    frequency = note_to_frequency(midi_number)
    note_signal = sine_wave(frequency, note_duration)
    note_envelope = adsr(note_duration, attack=0.1, decay=0.2, sustain=0.5, release=0.3)
    note_with_envelope = note_signal * note_envelope
    my_melody_audio.append(note_with_envelope.data)

my_full_melody = np.concatenate(my_melody_audio)
my_melody_signal = AudioSignal(my_full_melody, 44100)

display(play_sound(my_melody_signal, "オリジナルメロディー"))

## 📚 今日のまとめ

### 学んだこと
1. **クリック音の問題**: 音の急な開始・終了で発生
2. **エンベロープ**: 音量の時間的変化を制御
3. **ADSR**: Attack, Decay, Sustain, Release の4段階
4. **楽器の特徴**: エンベロープで楽器らしさを表現
5. **音楽的表現**: メロディーに自然さを与える

### 使ったライブラリ
- `linear_envelope()`: シンプルなフェード
- `adsr()`: 4段階エンベロープ
- `signal * envelope`: エンベロープの適用（AudioSignal の演算子）

### エンベロープの効果
- **技術的**: クリック音の除去
- **音楽的**: 楽器らしい表現
- **芸術的**: 音の感情表現

### 次回予告
次回は「**オシレーターと音色**」を学びます。
サイン波以外の波形（ノコギリ波、矩形波など）を使って、もっと豊かな音色を作ります！

---
**お疲れさまでした！** 🎉